In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
Mounted at /content/drive


In [ ]:
data = pd.read_csv('/content/drive/MyDrive/GSE115469_Data.csv', index_col=0)

In [ ]:
data.shape

(20007, 8444)

In [ ]:
data.isna().values.any()

np.False_

np.False_

# PRETPROCESIRANJE - Opsti koraci

Predlog za pretprocesiranje:

  - uklanjanje neinformativnih gena
    - geni koji se eksprimuju u manje od 5% celija
    - geni koji imaju skoro isti nivo ekspresije u svakoj celiji
  - clipping autlajera
  - transponovanje matrice

## Korak 1: Uklanjanje neinformativnih gena

### 1.1 Geni koji se eksprimuju u manje od 5% celija

In [ ]:
gene_frequency = (data > 0).mean(axis=1)
data = data.loc[gene_frequency >= 0.05]

In [ ]:
data.shape

(6717, 8444)

In [ ]:
# num_of_cells = data.shape[1]
# five_percent = np.floor((num_of_cells / 100) * 5)

In [ ]:
# def no_of_cells_exp(data, gene):
#   col = data.loc[gene]
#   return np.sum(col.astype(bool))

In [ ]:
# for gene in data.index:
#   if no_of_cells_exp(data, gene) <= five_percent:
#     data.drop(gene, axis=0)

### 1.2 Geni koji imaju skoro isti nivo ekspresije u svakoj celiji

In [ ]:
gene_var = data.var(axis=1)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:2000]

In [ ]:
gene_var_keep

,0
APOC3,12.340822
APOA2,11.126836
ORM1,9.849425
HP,9.393121
ALB,9.389534
...,...
TACC1,0.375015
PDXDC1,0.374714
FUNDC2,0.374669
RNF167,0.374545


In [ ]:
data = data.loc[gene_var_keep.index]

In [ ]:
data.shape

(2000, 8444)

## Korak 2: Outlier clipping

In [ ]:
data = data.clip(upper=data.quantile(0.999).max())

## Korak 3: Transponovanje matrice

In [ ]:
data = data.T

In [ ]:
data.shape

(8444, 2000)

(8444, 2000)

# PRETPROCESIRANJE - Pravila pridruzivanja

  - promena formata u transakcije -> Cell1 = {Gene1, Gene2, Gene5} <=> Cell1 = (1,1,0,0,1) (binarizacija)
    - gen je prisutan u celiji ako ekspresija prelazi odredjeni threshold (npr. 75. percentil gena)
  - uklanjanje previse cestih gena
  - uklanjanje previse retkih gena

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;APRIORI
- feature selection - izdvojiti najvarijabilnije gene, npr. 2000 njih

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;ECLAT
- vertical data layout

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;CHARM
- support threshold? (valjda je vec obradjeno)

In [ ]:
# kopija podataka za pravila pridruzivanja (association rules)
AR_data = data

In [ ]:
AR_data.head(10)

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,SAA1,APOA1,FGA,...,HMGCR,RNF149,GIMAP1,APH1A,MAF1,TACC1,PDXDC1,FUNDC2,RNF167,PSAT1
P1TLH_AAACCTGAGCAGCCTC_1,0.836165,0.836165,0.000000,1.362102,0.000000,0.836165,0.000000,1.362102,0.836165,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.836165,0.0,0.000000,0.0,0.0
P1TLH_AAACCTGTCCTCATTA_1,1.436498,1.436498,3.458904,1.149924,4.797971,2.059859,0.572995,2.493720,1.300315,1.675472,...,0.0,0.000000,0.000000,0.314760,0.0,0.982011,0.0,0.000000,0.0,0.0
P1TLH_AAACCTGTCTAAGCCA_1,1.820614,0.000000,0.000000,1.180248,0.000000,0.000000,1.180248,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,1.180248,0.0,1.180248,0.0,0.0
P1TLH_AAACGGGAGTAGGCCA_1,0.000000,0.644905,0.000000,0.000000,0.000000,0.000000,0.000000,1.428094,0.000000,0.000000,...,0.0,1.428094,0.644905,0.644905,0.0,0.000000,0.0,0.000000,0.0,0.0
P1TLH_AAACGGGGTTCGGGCT_1,1.284221,1.284221,0.780522,1.656843,1.656843,0.780522,0.000000,1.656843,0.780522,0.000000,...,0.0,0.000000,1.284221,0.780522,0.0,0.000000,0.0,0.000000,0.0,0.0
P1TLH_AAAGCAACAGTAAGAT_1,0.000000,2.216831,0.000000,2.216831,2.216831,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,2.216831,0.0,0.0
P1TLH_AAAGCAAGTCGCGTGT_1,3.072206,1.512300,1.512300,2.234284,1.512300,0.000000,2.234284,2.234284,1.512300,0.000000,...,0.0,1.512300,2.234284,0.000000,0.0,0.000000,0.0,1.512300,0.0,0.0
P1TLH_AAAGCAAGTGTTTGTG_1,1.172884,0.876891,1.172884,1.172884,1.628089,0.504068,0.000000,2.252131,0.876891,0.504068,...,0.0,0.876891,0.504068,0.504068,0.0,0.504068,0.0,0.000000,0.0,0.0
P1TLH_AAAGCAAGTTGATTCG_1,2.126138,0.000000,2.456096,2.456096,1.085305,1.697618,0.000000,3.610965,2.126138,1.697618,...,0.0,0.000000,1.085305,0.000000,0.0,1.085305,0.0,0.000000,0.0,0.0
P1TLH_AAAGTAGCAGACGTAG_1,0.775766,1.649062,1.649062,0.775766,1.277507,1.277507,0.000000,2.889257,1.277507,0.775766,...,0.0,0.000000,0.775766,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0


## Korak 1: Binarizacija

In [ ]:
gene_thresholds = AR_data.quantile(0.75, axis=1)
binary_AR_data = AR_data.gt(gene_thresholds, axis=0).astype(int)

In [ ]:
binary_AR_data.head(10)

,APOC3,APOA2,ORM1,HP,ALB,APOC1,TTR,SAA1,APOA1,FGA,...,HMGCR,RNF149,GIMAP1,APH1A,MAF1,TACC1,PDXDC1,FUNDC2,RNF167,PSAT1
P1TLH_AAACCTGAGCAGCCTC_1,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
P1TLH_AAACCTGTCCTCATTA_1,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,1,0,0,0,0
P1TLH_AAACCTGTCTAAGCCA_1,1,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,1,0,1,0,0
P1TLH_AAACGGGAGTAGGCCA_1,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
P1TLH_AAACGGGGTTCGGGCT_1,1,1,0,1,1,0,0,1,0,0,...,0,0,1,0,0,0,0,0,0,0
P1TLH_AAAGCAACAGTAAGAT_1,0,1,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
P1TLH_AAAGCAAGTCGCGTGT_1,1,0,0,1,0,0,1,1,0,0,...,0,0,1,0,0,0,0,0,0,0
P1TLH_AAAGCAAGTGTTTGTG_1,1,0,1,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
P1TLH_AAAGCAAGTTGATTCG_1,1,0,1,1,0,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
P1TLH_AAAGTAGCAGACGTAG_1,0,1,1,0,1,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


## Korak 2: Uklanjanje previse cestih/retkih gena

In [ ]:
gene_frequency = binary_AR_data.mean(axis=0)
print(gene_frequency.describe())

count    2000.000000
mean        0.184689
std         0.217804
min         0.019541
25%         0.056815
50%         0.096459
75%         0.203961
max         1.000000
dtype: float64


In [ ]:
# ove granice se mogu namestati
print((gene_frequency > 0.90).sum())
print((gene_frequency < 0.06).sum())

64
552


In [ ]:
binary_AR_data.shape

(8444, 2000)

In [ ]:
binary_AR_data = binary_AR_data.loc[:, (gene_frequency >= 0.06) & (gene_frequency <= 0.90)]

In [ ]:
binary_AR_data.shape

(8444, 1384)

# PRETPROCESIRANJE - Apriori